# 08a — Upload Phi-4 Adapters to Hugging Face Hub

Pushes the 6 Phi-4 adapters (3 LoRA + 3 QLoRA) from Drive to your HF account so the public demo Space can load them.

**HF username:** `Gbolahan507`

**Adapter repos created:**
- `Gbolahan507/phi4-lora-bbbp` ← from `adapters/phi4_bbbp_r8_lr1e-04`
- `Gbolahan507/phi4-lora-bace` ← from `adapters/phi4_bace_r8_lr1e-04`
- `Gbolahan507/phi4-lora-esol` ← from `adapters/phi4_esol_r8_lr1e-04`
- `Gbolahan507/phi4-qlora-bbbp` ← from `adapters/qlora_phi4_bbbp`
- `Gbolahan507/phi4-qlora-bace` ← from `adapters/qlora_phi4_bace`
- `Gbolahan507/phi4-qlora-esol` ← from `adapters/qlora_phi4_esol`

CPU runtime is fine — this is just file transfer.

## 1. Setup

In [ ]:
%pip install -q --upgrade huggingface_hub

In [ ]:
from huggingface_hub import login, HfApi, create_repo, upload_folder
from pathlib import Path

login()  # paste your hf_... token (Write scope) from huggingface.co/settings/tokens

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

ADAPTERS_DIR = Path('/content/drive/MyDrive/chemistry-peft-fyp/adapters')
print('Adapters root:', ADAPTERS_DIR)
print()
print('Contents:')
for p in sorted(ADAPTERS_DIR.iterdir()):
    print(' ', p.name)

## 2. Map local folders → HF repo IDs

Folder names from Phase 5/6 follow these patterns:
- LoRA Phi-4: `phi4_<ds>_r<rank>_lr<lr>`  (e.g. `phi4_bbbp_r8_lr1e-04`)
- QLoRA Phi-4: `qlora_phi4_<ds>`  (e.g. `qlora_phi4_bbbp`)

We use a glob so the exact rank/lr suffix doesn't matter.

In [ ]:
HF_USERNAME = 'Gbolahan507'
DATASETS = ['bbbp', 'bace', 'esol']

def find_lora_dir(ds):
    # match phi4_<ds>_r*_lr* (the Phase 5 best run)
    matches = sorted(ADAPTERS_DIR.glob(f'phi4_{ds}_r*_lr*'))
    return matches[0] if matches else None

def find_qlora_dir(ds):
    candidate = ADAPTERS_DIR / f'qlora_phi4_{ds}'
    return candidate if candidate.exists() else None

uploads = []
for ds in DATASETS:
    lora_dir  = find_lora_dir(ds)
    qlora_dir = find_qlora_dir(ds)
    uploads.append(('lora',  ds, lora_dir,  f'{HF_USERNAME}/phi4-lora-{ds}'))
    uploads.append(('qlora', ds, qlora_dir, f'{HF_USERNAME}/phi4-qlora-{ds}'))

print('Plan:')
for variant, ds, local, repo_id in uploads:
    status = 'FOUND' if local and local.exists() else 'MISSING'
    print(f'  [{status}] {variant} {ds} -> {repo_id}  ({local})')

## 3. Upload

In [ ]:
api = HfApi()

for variant, ds, local, repo_id in uploads:
    if local is None or not local.exists():
        print(f'[skip] {variant} {ds}: folder not found')
        continue

    print(f'\nCreating repo: {repo_id}')
    create_repo(repo_id, private=False, exist_ok=True)

    print(f'Uploading {local} -> {repo_id}')
    upload_folder(
        folder_path=str(local),
        repo_id=repo_id,
        commit_message=f'Add Phi-4 {variant.upper()} adapter for {ds.upper()}',
    )
    print(f'  done: https://huggingface.co/{repo_id}')

## 4. Verify

Click each printed link to confirm the adapters are visible on your HF profile.